# Treino do Lora em GPU no Kaggle

Treina o extrator de pedidos com o [Soup CLI](https://github.com/MakazhanAlpamys/Soup) numa GPU T4
e avalia no mesmo conjunto de teste do repositório.

Antes de rodar, no painel lateral **Settings**:

1. **Accelerator**: `GPU T4 x2`
2. **Internet**: ligada (precisa de telefone verificado na conta)

Para rodar sem deixar o navegador aberto, use **Save Version → Save & Run All (Commit)**.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Baixar o projeto e instalar o Soup

In [ ]:
!git clone -q https://github.com/FabioAGC/Lora.git
%cd Lora
!pip install -q "soup-cli[train]"

## 3. Treinar

Usa uma só das duas T4: o treino em duas GPUs exige configuração extra e não compensa para um treino curto.

Para testar o modelo de 7B, edite `soup_gpu.yaml` como explicado no comentário do arquivo.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
!soup train --config soup_gpu.yaml --yes

## 4. Avaliar no conjunto de teste (200 pedidos)

In [ ]:
%env PYTHONPATH=src
!python -m pedido_lora.avaliar --nome soup_gpu --modelo Qwen/Qwen2.5-1.5B-Instruct --adaptador outputs/soup_gpu

## 5. Testar uma mensagem

In [ ]:
!python -m pedido_lora.inferir --modelo Qwen/Qwen2.5-1.5B-Instruct --adaptador outputs/soup_gpu "me ve 2 xburguer sem cebola e uma coquinha zero, pago no pix"

## 6. Guardar o resultado

Tudo que fica em `/kaggle/working` aparece na aba **Output** do notebook depois do commit, para baixar.

In [ ]:
!rm -rf /kaggle/working/resultado && mkdir -p /kaggle/working/resultado
!cp -r outputs/soup_gpu results /kaggle/working/resultado/
!rm -rf /kaggle/working/resultado/soup_gpu/checkpoint-*
!ls -R /kaggle/working/resultado | head -30

## 7. Publicar no Hugging Face (opcional)

Crie um token de escrita no Hugging Face e guarde no Kaggle em **Add-ons → Secrets** com o nome `HF_TOKEN`.
Troque `SEU_USUARIO` pelo seu usuário.

O model card (`docs/MODEL_CARD.md`) descreve o modelo de 0,5B. A célula abaixo troca o modelo base,
mas a tabela de resultados do card precisa ser atualizada à mão com os números da etapa 4.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!sed -i 's#base_model: .*#base_model: Qwen/Qwen2.5-1.5B-Instruct#' docs/MODEL_CARD.md
!pip install -q huggingface_hub
!python -m pedido_lora.publicar --usuario SEU_USUARIO --adaptador outputs/soup_gpu